# Entender `w` y `b`: cómo cambian una predicción y su costo

Este notebook no presupone experiencia previa en estadística — solo álgebra de
secundaria. Seguro ya viste la ecuación de una recta como $y = mx + b$: en
machine learning se escribe casi igual, solo que a la pendiente $m$ la
llamamos $w$ (de *weight*, "peso"):

$$\hat{y}=wx+b$$

El sombrerito en $\hat y$ (se lee "y gorro") solo indica "esto es una
**predicción**, no el valor real". La pregunta de este notebook es: **¿por qué
una regresión lineal necesita ajustar `w` y `b`, y qué le pasa a la predicción
cuando cambian?**

Usaremos un ejemplo de horas de estudio (`x`) y calificación (`y`).

## 1. Qué representa cada parámetro

Imagina que la recta es una regla de plástico apoyada sobre la gráfica:

- **`w` (peso o pendiente)** — en la ecuación $\hat y = wx + b$, es el número
  que multiplica a $x$. Controla la inclinación: si `w` aumenta, la recta sube
  más rápido al avanzar hacia la derecha. Con una única variable de entrada,
  también se le llama simplemente **pendiente**, igual que en la ecuación de
  secundaria.
- **`b` (bias o sesgo)** — es el número que se suma al final. Mueve toda la
  recta hacia arriba o hacia abajo, sin inclinarla. Se llama **intercepto**
  porque es el punto donde la recta corta el eje vertical: fíjate que si
  $x=0$, la fórmula queda $\hat y = w \cdot 0 + b = b$. Por eso $b$ es "la
  predicción cuando $x$ vale cero".

No son intercambiables: cambiar `w` **rota** la regla alrededor de un punto;
cambiar `b` la **desliza** verticalmente sin rotarla. Vamos a verlo, no solo a
leerlo.

In [1]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

# Datos didácticos: cada hora extra aumenta la calificación observada en 3 puntos.
datos = pl.DataFrame({
    "horas_estudio": [0, 1, 2, 3, 4, 5, 6],
    "calificacion_real": [4, 7, 10, 13, 16, 19, 22],
})

def predecir(x, w, b):
    return w * np.asarray(x) + b

def mse(y_real, y_predicho):
    return np.mean((np.asarray(y_real) - np.asarray(y_predicho)) ** 2)

datos

horas_estudio,calificacion_real
i64,i64
0,4
1,7
2,10
3,13
4,16
5,19
6,22


## 2. Solo `b`: deslizar la recta

Antes de tocar `w`, dejemos la pendiente fija en un valor cualquiera (`w=3`) y
movamos únicamente `b`. Si la idea de "intercepto" es correcta, todas las
rectas deberían verse igual de inclinadas, solo que unas más arriba y otras más
abajo.

In [2]:
x = datos["horas_estudio"].to_numpy()
y = datos["calificacion_real"].to_numpy()

valores_b_demo = [-2.0, 4.0, 10.0, 16.0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="datos reales", marker={"size": 10, "color": "black"}))
for b_demo in valores_b_demo:
    fig.add_trace(go.Scatter(
        x=x, y=predecir(x, 3.0, b_demo), mode="lines",
        name=f"b={b_demo:.0f}", line={"width": 2},
    ))
fig.update_layout(
    title="Con w fijo en 3, cambiar b solo desliza la recta hacia arriba o abajo",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

Todas las líneas son **paralelas**: la misma inclinación, distinta altura.
Ahora hagamos lo opuesto — fijar `b` y mover solo `w` — para confirmar que la
pendiente es lo único que cambia la inclinación.

In [3]:
valores_w_demo = [0.5, 1.5, 3.0, 5.5]
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="datos reales", marker={"size": 10, "color": "black"}))
for w_demo in valores_w_demo:
    fig.add_trace(go.Scatter(
        x=x, y=predecir(x, w_demo, 4.0), mode="lines",
        name=f"w={w_demo}", line={"width": 2},
    ))
fig.update_layout(
    title="Con b fijo en 4, cambiar w rota la recta alrededor de x=0",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

Ahora todas las líneas **arrancan del mismo punto** (en $x=0$, todas valen
$b=4$) pero se abren en abanico con distinta inclinación. Eso es literalmente
lo que hace `w`: decide cuánto sube la predicción por cada unidad que aumenta
`x`.

## 3. Cuatro combinaciones, cuatro comportamientos

En estos datos, la regla perfecta es $\hat y = 3x + 4$. No hace falta
adivinarla: en la vida real el algoritmo la busca minimizando el costo. Por
ahora la usamos como referencia para contrastar contra combinaciones
incorrectas:

- **Referencia:** `w = 3`, `b = 4` — coincide con los datos.
- **Peso demasiado pequeño:** `w = 1.5`, `b = 4` — arranca bien pero se queda
  corta cada vez más; la recta es demasiado plana.
- **Bias demasiado alto:** `w = 3`, `b = 12` — tiene la inclinación correcta,
  pero toda la recta quedó desplazada 8 puntos hacia arriba.
- **Ambos incorrectos:** `w = 4.5`, `b = -2` — empieza demasiado abajo y sube
  demasiado rápido.

In [4]:
modelos = [
    {"nombre": "referencia: w=3, b=4", "w": 3.0, "b": 4.0, "color": "#2ca02c"},
    {"nombre": "w pequeño: w=1.5, b=4", "w": 1.5, "b": 4.0, "color": "#ff7f0e"},
    {"nombre": "b alto: w=3, b=12", "w": 3.0, "b": 12.0, "color": "#d62728"},
    {"nombre": "ambos: w=4.5, b=-2", "w": 4.5, "b": -2.0, "color": "#9467bd"},
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y, mode="markers", name="datos reales",
    marker={"size": 10, "color": "black"},
))
for modelo in modelos:
    fig.add_trace(go.Scatter(
        x=x, y=predecir(x, modelo["w"], modelo["b"]), mode="lines",
        name=modelo["nombre"], line={"color": modelo["color"]},
    ))
fig.update_layout(
    title="Cambiar w inclina la recta; cambiar b la desplaza",
    xaxis_title="Horas de estudio", yaxis_title="Calificación",
)
fig.show()

## 4. El costo pone un número al error

Mirar la gráfica ya te dice cuál recta parece mejor, pero "parece mejor" no es
algo que un algoritmo pueda calcular. Necesitamos un número. Usamos el **error
cuadrático medio (MSE)**, que ya viste (o verás) en detalle en
[`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb):

$$MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2$$

En corto: por cada punto, mide qué tan lejos quedó la predicción del valor
real, eleva esa distancia al cuadrado (para que los errores grandes pesen más
y el signo no esconda nada) y promedia. El MSE es 0 si todas las predicciones
coinciden exactamente con los datos; cuanto mayor sea, peor encaja la recta.

In [5]:
comparacion = pl.DataFrame([
    {
        "modelo": modelo["nombre"], "w": modelo["w"], "b": modelo["b"],
        "mse": mse(y, predecir(x, modelo["w"], modelo["b"])),
    }
    for modelo in modelos
]).sort("mse")
comparacion

modelo,w,b,mse
str,f64,f64,f64
"""referencia: w=3, b=4""",3.0,4.0,0.0
"""ambos: w=4.5, b=-2""",4.5,-2.0,11.25
"""w pequeño: w=1.5, b=4""",1.5,4.0,29.25
"""b alto: w=3, b=12""",3.0,12.0,64.0


Observa los dos errores de una forma intuitiva mirando los residuos (la
distancia vertical entre cada punto y cada recta incorrecta):

- Con `w` incorrecto, el error **depende de `x`**: cerca de 0 puede ser
  pequeño, pero crece al alejarse — la inclinación no describe bien la
  relación.
- Con `b` incorrecto, el error es aproximadamente el **mismo desplazamiento
  vertical** para todos los puntos — la forma es correcta, pero la recta está
  mal colocada.

In [6]:
residuos = pl.concat([
    datos.select("horas_estudio").with_columns(
        pl.Series("residuo", y - predecir(x, 1.5, 4.0)),
        pl.lit("w pequeño").alias("caso"),
    ),
    datos.select("horas_estudio").with_columns(
        pl.Series("residuo", y - predecir(x, 3.0, 12.0)),
        pl.lit("b alto").alias("caso"),
    ),
])
fig = px.bar(
    residuos, x="horas_estudio", y="residuo", color="caso", barmode="group",
    title="El error por w cambia con x; el error por b desplaza todos los puntos por igual",
)
fig.add_hline(y=0, line_color="black")
fig.show()

Por eso ajustar solo uno de los dos parámetros normalmente no basta: una
predicción eficaz necesita tanto la inclinación (`w`) como la posición (`b`)
adecuadas.

## 5. Barrer muchos valores de `w` a la vez

En vez de comparar cuatro rectas sueltas, veamos qué pasa si probamos **muchos**
valores de `w` (con `b=4` fijo) y coloreamos cada recta según su costo: las
rectas cercanas al valor correcto deberían verse en un color, y las alejadas en
otro.

In [7]:
import plotly.colors as pc

valores_w_barrido = np.linspace(0.5, 6.0, 12)
costos_barrido = np.array([mse(y, predecir(x, w_c, 4.0)) for w_c in valores_w_barrido])
costo_normalizado = (costos_barrido - costos_barrido.min()) / (costos_barrido.max() - costos_barrido.min())
colores_barrido = pc.sample_colorscale("Viridis_r", costo_normalizado)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="datos reales", marker={"size": 11, "color": "black"}))
for w_c, color_c in zip(valores_w_barrido, colores_barrido):
    fig.add_trace(go.Scatter(
        x=x, y=predecir(x, w_c, 4.0), mode="lines", showlegend=False,
        line={"color": color_c, "width": 2},
    ))
fig.update_layout(title="Barrido de w (b=4 fijo): el color va del mejor ajuste (oscuro) al peor (claro)")
fig.show()

Ahora convirtamos esa misma idea en la gráfica que realmente importa para
entrenar un modelo: en el eje horizontal ya no ponemos "horas de estudio", sino
el propio valor de `w`, y en el eje vertical ponemos el MSE de esa recta.
Primero dejamos `b = 4` fijo y probamos muchos valores de `w`. Después dejamos
`w = 3` fijo y probamos muchos valores de `b`. Cada gráfica tiene forma de
valle: el punto más bajo es el menor costo posible bajo esa condición.

In [8]:
valores_w = np.linspace(-1, 7, 161)
valores_b = np.linspace(-6, 14, 161)

costo_por_w = pl.DataFrame({
    "w": valores_w,
    "mse": [mse(y, predecir(x, w_candidato, 4)) for w_candidato in valores_w],
})
costo_por_b = pl.DataFrame({
    "b": valores_b,
    "mse": [mse(y, predecir(x, 3, b_candidato)) for b_candidato in valores_b],
})

fig_w = px.line(costo_por_w, x="w", y="mse", title="Costo al cambiar w (b fijo en 4)")
fig_w.add_vline(x=3, line_dash="dash", annotation_text="mejor w")
fig_w.show()

fig_b = px.line(costo_por_b, x="b", y="mse", title="Costo al cambiar b (w fijo en 3)")
fig_b.add_vline(x=4, line_dash="dash", annotation_text="mejor b")
fig_b.show()

## 6. Ajustar `w` y `b` juntos

En realidad el algoritmo prueba ambos parámetros a la vez, no uno primero y
otro después. Podemos imaginar todos los pares posibles `(w, b)` como un mapa:
cada color representa un MSE. La zona más oscura es el mejor par.

El entrenamiento consiste en moverse por este mapa hacia abajo — el mecanismo
exacto (descenso de gradiente) se explica paso a paso en
[`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb).

In [9]:
rejilla_w = np.linspace(0, 6, 81)
rejilla_b = np.linspace(-4, 12, 81)
superficie = pl.DataFrame([
    {"w": w_candidato, "b": b_candidato, "mse": mse(y, predecir(x, w_candidato, b_candidato))}
    for w_candidato in rejilla_w
    for b_candidato in rejilla_b
])

fig = px.density_heatmap(
    superficie, x="w", y="b", z="mse", histfunc="avg",
    color_continuous_scale="Viridis_r",
    title="Mapa de costo: el mínimo está cerca de w=3 y b=4",
)
fig.add_trace(go.Scatter(x=[3], y=[4], mode="markers", name="mejor combinación", marker={"color": "red", "size": 11}))
fig.show()

## 7. Lo esencial para recordar

1. `w` decide **cuánto cambia la predicción** cuando cambia la entrada; es la
   pendiente, igual que la $m$ de $y=mx+b$ en álgebra.
2. `b` decide **desde qué altura parte** la predicción; es el desplazamiento
   vertical, el valor de $\hat y$ cuando $x=0$.
3. Una combinación equivocada de `w` y `b` produce predicciones alejadas de los
   datos, y el patrón del error (¿depende de `x` o es constante?) delata cuál
   parámetro está mal.
4. El MSE convierte esos errores en un único número que se puede comparar y
   minimizar.
5. Entrenar el modelo es encontrar los valores de `w` y `b` con menor costo en
   los datos de entrenamiento.

**Prueba tú:** cambia los valores de `modelos` en la sección 3 y vuelve a
ejecutar las gráficas. Antes de mirar el MSE, intenta predecir qué ocurrirá si
duplicas `w` o si sumas 5 a `b`. Luego revisa si tu predicción coincidió con la
gráfica de residuos.